# Acceptance: lazy-idiom bridges

**This is a sign-off page, not a tutorial.** It exercises roadmap step 31, which closes
composability finding `CX26`: a likelihood scale (`NormalLikelihood(sd=...)` and
`NegativeBinomial(dispersion=...)`) accepts any parameter-only rate expression, not only a
bare `Param`. The claim the reader checks is that an expression scores exactly like the float
it evaluates to. The automated checks live in `tests/test_epi_calibration.py`.

The model is a seasonal SEIRS in two age groups, run for two years with weekly saves. Later
sections (B–E, added by roadmap steps 32–35) reuse this same model. Run every cell top to
bottom (a few seconds). Sign off only if every **What to check** box holds.

| | You are checking that… |
|---|---|
| A | a negative-binomial `dispersion` written as an expression of a parameter scores exactly like the float it evaluates to, and the likelihood peaks near the true value |

Later steps add sections B–E to this page.

## The model: a seasonal SEIRS in two age groups

Two age groups mix assortatively. The infection rate is `beta * (1 + amp * sin(2π t / 365))`,
built as a rate expression of `Time()`, so the seasonal forcing is part of the model and not a
number that was precomputed. Recovered people lose immunity at rate `omega` (`R → S`, mean
duration 120 days). Without waning, the first wave burns out in about 40 of the 104 weeks and
leaves little signal for the dispersion. Incidence is the infection flow summed over age groups
and saved weekly with `FlowMass(...).total().midpoint()`.

In [ ]:
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    Time,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import NegativeBinomial

state = Property("state", ("S", "E", "I", "R"))
age = Property("age", ("young", "old"))
pmap = PropertyMap.from_property(state).stratify(age)

beta = Param("beta") * (1 + Param("amp") * np.sin(2 * np.pi * Time() / 365))
mixing = MixingMatrix(
    age, np.array([[0.8, 0.2], [0.3, 0.7]]), normalize="rows", check_reciprocal=False
)
model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["E"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=mixing.prop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=beta,
        ),
    )
)
model.add_flow(TransitionFlow("progression", state["E"], state["I"], Param("sigma")))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
model.add_flow(TransitionFlow("waning", state["R"], state["S"], Param("omega")))
compiled = model.compile()

y0 = np.zeros(pmap.size)
y0[pmap.select(state["S"] & age["young"])] = 79990.0
y0[pmap.select(state["I"] & age["young"])] = 10.0
y0[pmap.select(state["S"] & age["old"])] = 20000.0

TRUE_PARAMS = {"beta": 0.5, "amp": 0.3, "sigma": 0.2, "gamma": 1.0 / 7.0, "omega": 1.0 / 120.0}
weeks = np.arange(7.0, 729.0, 7.0)

outputs = OutputSet()
outputs["incidence"] = FlowMass("infection").total().midpoint()
solved = compiled.run(
    TRUE_PARAMS,
    y0,
    t0=0.0,
    t1=728.0,
    dt=0.25,
    save=outputs.plan(SavePlan(ts=weeks)),
    solver="euler",
)
named = outputs.evaluate(solved, TRUE_PARAMS)
predicted = np.asarray(named["incidence"].values).reshape(-1)

## A. A likelihood scale can be an expression

The **true** dispersion is `k = exp(log_k)` with `log_k = 1.5`. The first code cell draws the
observed weekly counts from a negative binomial around the model's incidence with a fixed seed.
The figure plots the model incidence and the noisy counts. The counts should scatter around the
model curve with a spread that grows with the level.

The second code cell builds `NegativeBinomial(dispersion=np.exp(Param("log_k")))` and evaluates
its `log_prob` at each `log_k` on a grid from 0 to 3. It does the same with a float dispersion,
`np.exp(log_k)`, computed at that grid point. The figure plots both curves.

In [ ]:
TRUE_LOG_K = 1.5
rng = np.random.default_rng(31)
k_true = float(np.exp(TRUE_LOG_K))
observed = rng.negative_binomial(k_true, k_true / (k_true + predicted)).astype(float)

weekly = pd.DataFrame(
    {"model incidence": predicted, "observed (negative-binomial draw)": observed},
    index=pd.Index(weeks, name="day"),
)
fig_data = weekly.plot(
    title="A. Weekly incidence: the model and the noisy observations",
    labels={"index": "day", "value": "infections per week"},
)
fig_data.show()

In [ ]:
grid = np.linspace(0.0, 3.0, 61)
expression_likelihood = NegativeBinomial(dispersion=np.exp(Param("log_k")))
expression_ll = []
float_ll = []
for log_k in grid:
    expression_ll.append(
        float(
            expression_likelihood.log_prob(
                jnp.asarray(observed), jnp.asarray(predicted), {"log_k": log_k}
            )
        )
    )
    float_likelihood = NegativeBinomial(dispersion=float(np.exp(log_k)))
    float_ll.append(
        float(float_likelihood.log_prob(jnp.asarray(observed), jnp.asarray(predicted), {}))
    )
expression_ll = np.asarray(expression_ll)
float_ll = np.asarray(float_ll)

curves = pd.DataFrame(
    {
        "Param expression: np.exp(Param('log_k'))": expression_ll,
        "float dispersion: np.exp(log_k)": float_ll,
    },
    index=pd.Index(grid, name="log_k"),
)
fig_ll = curves.plot(
    title="A. Log-likelihood against log_k: the expression and the float",
    labels={"index": "log_k", "value": "mean log-likelihood per week"},
)
fig_ll.data[1].update(line={"color": "black", "dash": "dot", "width": 2})
fig_ll.add_vline(x=TRUE_LOG_K, line_dash="dash", annotation_text="true log_k")
fig_ll.show()

pd.Series(
    {
        "true log_k": TRUE_LOG_K,
        "grid maximum log_k": round(float(grid[np.argmax(expression_ll)]), 2),
        "max |expression - float| over the grid": float(
            np.max(np.abs(expression_ll - float_ll))
        ),
    },
    name="A",
    dtype=object,
)

> **What to check**
>
> - In the first figure, the noisy observations scatter around the model incidence curve, and
>   the scatter widens where the incidence is high.
> - In the second figure, the solid curve (dispersion written as `np.exp(Param("log_k"))`) and
>   the dotted curve (a float `np.exp(log_k)` at each point) lie **on top of each other** across
>   the whole grid.
> - The maximum of the log-likelihood lies **within about 0.3 of the true `log_k` = 1.5**
>   (the dashed line). It is a single weekly series, so the peak is noisy; across seeds it
>   lands within 0.3 for most draws, not all.
> - The table's "max |expression - float|" is at float32 rounding (far below 1e-4).